[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_83_Phase10_RAG_Production_Baseline.ipynb)

# Lesson 83 — Phase 10 Opener: RAG at Production Scale

### The naive baseline, and exactly where it breaks

> **Phase 9 is shipped.** You now own four cross-referencing open-source tools —
> `paper-distiller` (an agent), `agent-bench` (offline eval), `agent-obs` (production ops),
> and `orchestra-agents` (multi-agent orchestration). Phase 10 turns to the other half of a
> serious agent: **giving it access to *your* knowledge.**

**Phase 10 = RAG at production scale.** You met *intro* RAG all the way back in Lesson 7 and
vector databases in Lesson 20. This phase is different: we treat RAG as a **retrieval system
with a quality bar**, measure it honestly, and fix the specific ways it fails in production.

Today's lesson is the **opener**. We do one thing thoroughly:

1. Build a **complete naive RAG pipeline** end to end — load -> chunk -> embed -> index -> retrieve -> generate.
2. Put a **labeled question set** in front of it and **measure** retrieval and answer quality.
3. Expose the **four failure modes** that every naive RAG has — each one becomes a lesson in Phase 10.

By the end you'll understand the single most important idea in production RAG:

> **Building a RAG pipeline is easy. Retrieving the _right_ chunk is the entire game.**
> Most "our RAG is bad" problems are *retrieval* problems wearing a generation costume.

*Runs fully offline in Colab — no API key required. We use TF-IDF vectors so there are zero
downloads; the retrieval mechanics are identical to dense embeddings, and we mark exactly where
you'd swap in a real embedding model.*

## Where this sits — Phase 10 roadmap

| Lesson | Topic | The failure it fixes |
|---|---|---|
| **83 (today)** | **Naive RAG baseline + measurement** | *establishes the baseline & names the failures* |
| 84 | Chunking strategies | answers split across chunk boundaries; wrong granularity |
| 85 | Dense & hybrid retrieval | the **semantic gap** (synonyms lexical search can't match) |
| 86 | Reranking (cross-encoders) | right chunk retrieved but **not ranked #1** |
| 87 | Grounding, citations & abstention | answering confidently from **irrelevant** context |
| 88 | RAG evaluation (retrieval + faithfulness metrics) | "it feels better" with no numbers |
| 89 (capstone) | Ship a production RAG service | all of the above, packaged & deployed |

*Everything we discover breaking today has a dedicated fix later. The opener's job is to make
those failures **visible and measurable**, so the rest of the phase is fixing real numbers — not vibes.*

## Setup

One dependency: `scikit-learn` (for TF-IDF vectors + cosine similarity) and `numpy`. The
`anthropic` import is **optional** — if you add an `ANTHROPIC_API_KEY` Colab secret, the
generator will use a real LLM; otherwise it uses a deterministic grounded stub so the whole
notebook still runs.

In [ ]:
# In Colab this installs into the runtime. Locally it's a no-op if already present.
!pip install scikit-learn numpy -q

import re, math
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# --- Optional real LLM (safe to skip). Uses Colab Secrets if present. ---
CLIENT = None
try:
    from anthropic import Anthropic
    from google.colab import userdata          # only exists in Colab
    key = userdata.get("ANTHROPIC_API_KEY")
    if key:
        CLIENT = Anthropic(api_key=key)
        print("Real LLM generator ENABLED (claude via ANTHROPIC_API_KEY).")
except Exception:
    pass
if CLIENT is None:
    print("Running with the OFFLINE grounded stub (no API key needed). All cells still work.")


## §1 — Why RAG exists: the frozen-model problem

An LLM's weights are frozen at its training cutoff. It has never seen:

- your company's internal docs,
- last week's incident report,
- the private policy your support team follows.

Ask it a question about any of those and it does one of two bad things: **refuses**, or worse,
**confidently makes something up**. RAG (Retrieval-Augmented Generation) fixes this by, *for each
question*, retrieving the most relevant passages from **your** corpus and placing them in the
prompt — so the model answers from **evidence you supplied**, not from memory.

Below is our private corpus: a fictional SaaS company, **Nimbus**. None of these facts are in any
model's training data — which is the whole point.

In [ ]:
# A small PRIVATE knowledge base. Realistic multi-fact docs so chunking & ranking matter.
DOCS = {
 "refunds": ("Nimbus Refund Policy. Customers on the monthly plan may request a full refund "
   "within 14 days of any charge. Annual plans are refundable on a prorated basis for the "
   "remaining unused months. Refunds are issued to the original payment method and take 5 to "
   "10 business days to appear. One-time setup fees are non-refundable."),
 "ratelimits": ("Nimbus API Rate Limits. The Free tier allows 60 requests per minute. The Pro "
   "tier allows 600 requests per minute. The Enterprise tier allows 6000 requests per minute. "
   "Exceeding your limit returns HTTP 429. Each 429 response includes a Retry-After header "
   "telling you how many seconds to wait before retrying."),
 "sso": ("Nimbus Single Sign-On. SSO is available on the Enterprise plan only. We support SAML "
   "2.0 and OIDC. To configure SAML, an administrator uploads the identity provider metadata XML "
   "in the Security settings page. Just-in-time user provisioning is enabled by default so new "
   "users are created on first login."),
 "retention": ("Nimbus Data Retention. Application logs are retained for 30 days. Deleted "
   "projects are held in a recoverable trash state for 90 days before permanent deletion. "
   "Customers on the Enterprise plan can configure a custom retention window of up to 7 years "
   "for compliance."),
 "security": ("Nimbus Security. All customer data is encrypted at rest using AES-256 and in "
   "transit using TLS 1.3. Nimbus is SOC 2 Type II certified. Access to production systems "
   "requires hardware security keys. We run third-party penetration tests twice per year."),
 "credentials": ("Nimbus Account Access. If you are locked out, use the credential recovery flow "
   "on the sign-in page: enter your email and we send a one-time link that lets you set a new "
   "secret. Links expire after 30 minutes. Enabling two-factor authentication is strongly "
   "recommended for all accounts."),
 "pricing": ("Nimbus Pricing. The Free tier costs nothing and includes one project. The Pro tier "
   "costs 49 dollars per month and includes ten projects. The Enterprise tier is custom-priced "
   "and includes unlimited projects, SSO, and a dedicated support manager."),
 "support": ("Nimbus Support SLAs. Free tier support is community-only. Pro tier guarantees a "
   "first response within one business day. Enterprise tier guarantees a first response within "
   "one hour for urgent issues, twenty-four hours a day, seven days a week."),
}
print(f"{len(DOCS)} documents in the corpus.")

# A frozen model with NO retrieval simply cannot know these private facts:
def no_retrieval_answer(question):
    return "I don't have information about that."   # honest frozen model
print("No-retrieval baseline:", no_retrieval_answer("what is the Pro tier rate limit"))
# EXPERIMENT: this is the ceiling we must beat. Zero private questions answerable.


## §2 — Anatomy of a RAG pipeline

Every RAG system, from a weekend hack to one serving millions, is these six stages:

```
                 INDEX TIME (once, offline)               QUERY TIME (per question)
        +-----------------------------------+     +--------------------------------------+
 docs ->| 1. CHUNK  -> 2. EMBED -> 3. INDEX  |     | 4. RETRIEVE top-k -> 5. GENERATE     |
        |    split      vectorize   store    |     |    (embed query,     (LLM answers    |
        |    text       each chunk  vectors  |     |     cosine search)    from context)  |
        +-----------------------------------+     +--------------------------------------+
                                                              ^
                                                    6. (the honest part) MEASURE
```

Two of these stages decide most of your quality — and neither is the LLM:

- **Chunk** (stage 1): the wrong chunk size splits an answer in half or drowns it in noise.
- **Retrieve** (stage 4): if the right chunk isn't in the top-k, no amount of prompt engineering
  in stage 5 can save you. *The generator can only be as good as what retrieval hands it.*

We'll build each stage, then measure the pipeline as a whole.

## §3 — Stage 1: Chunking

You don't embed whole documents — you embed **chunks**. Retrieval returns whole chunks into the
prompt, so you want each unit **small enough to be precise** (mostly about one thing) but **large
enough to be self-contained** (the answer isn't cut off).

Naive chunking splits every N characters blindly and slices sentences in half. We do slightly
better with **sentence-aware chunking + overlap**: pack whole sentences up to a size budget, and
repeat the last sentence into the next chunk so a fact straddling a boundary survives.

*This is still a baseline — chunking strategy is the whole subject of Lesson 84. Today we just want
a reasonable default so we can see the rest of the pipeline.*

In [ ]:
def sent_split(text):
    # split on sentence-ending punctuation followed by whitespace
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text.strip()) if s.strip()]

def chunk_document(doc_id, text, max_chars=180, overlap_sents=1):
    # Sentence-aware packing with sentence overlap. Returns a list of chunk dicts.
    sents = sent_split(text)
    out, idx, k = [], 0, 0
    while k < len(sents):
        buf, length, start = [], 0, k
        while k < len(sents) and (length + len(sents[k]) <= max_chars or not buf):
            buf.append(sents[k]); length += len(sents[k]) + 1; k += 1
        out.append({"id": f"{doc_id}#{idx}", "doc": doc_id, "text": " ".join(buf)})
        idx += 1
        if overlap_sents and k < len(sents):
            k = max(k - overlap_sents, start + 1)   # step back for overlap
    return out

CHUNKS = []
for d, t in DOCS.items():
    CHUNKS.extend(chunk_document(d, t))

print(f"{len(DOCS)} docs -> {len(CHUNKS)} chunks (max_chars=180, 1-sentence overlap)\n")
for c in CHUNKS[:4]:
    print(f"  {c['id']:14} {c['text'][:64]}...")
# EXPERIMENT: bump max_chars to 400 or drop to 60 and re-run. Watch the chunk count
# swing. In section 8 we MEASURE how this single number moves retrieval accuracy.


## §4 — Stages 2 & 3: Embed + Index

To find relevant chunks we turn each chunk into a **vector**, then at query time turn the question
into a vector and return the chunks whose vectors are **closest** (cosine similarity).

Here we use **TF-IDF** vectors: fully offline, zero downloads, and the retrieval math (vectorize ->
cosine -> top-k) is *exactly* what a production system does. The **only** difference is vector
quality — TF-IDF vectors are sparse and **lexical** (they match shared words), whereas production
uses **dense** neural embeddings that match on *meaning*. That difference is not cosmetic: it's a
specific failure mode we catch in section 8 and fix in Lesson 85.

> **Swap point for production.** Replace the two `TfidfVectorizer` lines with a real embedder —
> `sentence-transformers` (open, local), or OpenAI / Cohere / Voyage embeddings — and keep the rest
> of the class identical. The interface `search(query, k)` does not change.

In [ ]:
class VectorIndex:
    # A minimal cosine-similarity vector index. Swap the vectorizer for dense embeddings in prod.
    def __init__(self, chunks):
        self.chunks = chunks
        self.vec = TfidfVectorizer(stop_words="english")
        self.M = self.vec.fit_transform([c["text"] for c in chunks])   # (n_chunks, vocab)

    def search(self, query, k=3):
        sims = cosine_similarity(self.vec.transform([query]), self.M)[0]
        order = np.argsort(-sims)[:k]
        return [(self.chunks[i], float(sims[i])) for i in order]

index = VectorIndex(CHUNKS)

print("Query: 'how long do refunds take'  -> top 3 chunks:")
for c, s in index.search("how long do refunds take", k=3):
    print(f"  sim={s:.3f}  {c['id']:12} {c['text'][:56]}")
# EXPERIMENT: try 'refund window' vs 'money back'. TF-IDF rewards SHARED WORDS, so
# 'money back' (no shared words with the doc) scores far lower. Remember that.


## §5 — Stage 5: Grounded generation

Retrieval hands the generator a set of chunks. The generator's job is to answer **only** from those
chunks, and to **abstain** when the answer isn't there. That grounding rule is what separates RAG
from an LLM guessing — and it's a system-prompt discipline, not magic:

> *"Answer ONLY using the provided context. If the answer is not in the context, say you don't know.
> Cite the chunk ids you used."*

Our generator works **with or without** an API key. With a key it calls a real model; without one it
uses a deterministic **grounded stub** that returns the top retrieved chunk as the answer context
and **refuses when the top similarity is below a floor** — a stand-in for a real model reading that
context. Same interface either way.

In [ ]:
GROUNDED_SYS = ("Answer ONLY using the provided context. If the answer is not in the context, "
    "reply exactly: I don't know from the provided documents. Cite the chunk ids you used.")

def format_context(retrieved):
    return "\n".join(f"[{c['id']}] {c['text']}" for c, _ in retrieved)

def generate(question, retrieved, min_sim=0.10, client=CLIENT):
    # No relevant context? Abstain instead of hallucinating.
    if not retrieved or retrieved[0][1] < min_sim:
        return {"text": "I don't know from the provided documents.", "cites": []}
    if client is None:
        # OFFLINE grounded stub: hand back the single most-relevant chunk as the answer context.
        top, _ = retrieved[0]
        return {"text": top["text"], "cites": [top["id"]]}
    # REAL LLM path (used only if you set an API key).
    msg = client.messages.create(
        model="claude-sonnet-5", max_tokens=300, system=GROUNDED_SYS,
        messages=[{"role": "user",
                   "content": f"Context:\n{format_context(retrieved)}\n\nQuestion: {question}"}])
    return {"text": msg.content[0].text, "cites": [c["id"] for c, _ in retrieved]}

# In-scope question -> grounded answer:
r = index.search("what encryption protects data at rest", k=3)
print("ANSWER :", generate("what encryption protects data at rest", r)["text"][:88])

# Out-of-corpus question with a high floor -> abstains instead of inventing:
r2 = index.search("what is the CEO's home address", k=3)
print("ABSTAIN:", generate("what is the CEO's home address", r2, min_sim=0.9)["text"])
# EXPERIMENT: set min_sim=0.0 on that second call. It will now ANSWER from an
# irrelevant chunk. That is failure mode #3 — hold that thought for section 8.


## §6 — The whole pipeline in one object

Now wrap all six stages behind one `answer(question)` call. This is the naive RAG baseline we spend
the rest of Phase 10 improving.

In [ ]:
class RagPipeline:
    def __init__(self, docs, k=3, min_sim=0.10, max_chars=180):
        self.k, self.min_sim = k, min_sim
        chunks = []
        for d, t in docs.items():
            chunks.extend(chunk_document(d, t, max_chars=max_chars))
        self.chunks = chunks
        self.index = VectorIndex(chunks)

    def answer(self, question):
        retrieved = self.index.search(question, k=self.k)
        out = generate(question, retrieved, self.min_sim)
        out["retrieved"] = [(c["id"], round(s, 3)) for c, s in retrieved]
        return out

rag = RagPipeline(DOCS, k=3)
for q in ["what is the Pro tier rate limit", "how much does the Pro plan cost per month"]:
    a = rag.answer(q)
    print(f"Q: {q}\n   -> {a['text'][:80]}\n   retrieved: {a['retrieved']}\n")


## §7 — Now MEASURE it (the part most tutorials skip)

A demo that answers two cherry-picked questions tells you nothing. To know whether RAG *works* you
need a **labeled evaluation set**: questions paired with the document that truly answers them
(`gold`) and a fact the answer must contain (`expect`). Then compute:

- **hit@k** — did retrieval put the gold document in the top *k*? *(Pure retrieval quality.)*
- **MRR** — mean reciprocal rank of the gold doc. *(Rewards ranking it near the top.)*
- **answer accuracy** — does the final answer contain the expected fact? *(End-to-end quality.)*

We deliberately include **hard** questions: a vocabulary-mismatch question, a distractor, and an
**out-of-scope** question with no answer in the corpus. That's not sabotage — production traffic is
*full* of these.

In [ ]:
EVAL = [
 {"q":"how many days to get a refund on a monthly plan","gold":"refunds","expect":"14"},
 {"q":"what is the Pro tier rate limit","gold":"ratelimits","expect":"600"},
 {"q":"which plans include single sign-on","gold":"sso","expect":"Enterprise"},
 {"q":"how long are deleted projects recoverable","gold":"retention","expect":"90"},
 {"q":"what encryption is used for data at rest","gold":"security","expect":"AES-256"},
 {"q":"how much does the Pro plan cost per month","gold":"pricing","expect":"49"},
 {"q":"how fast does Enterprise support respond to urgent issues","gold":"support","expect":"one hour"},
 {"q":"what is the maximum custom retention window for compliance","gold":"retention","expect":"7 years"},
 {"q":"how do I reset my password","gold":"credentials","expect":"link"},        # SEMANTIC GAP
 {"q":"does Nimbus offer a mobile app for iOS","gold":None,"expect":"I don't know"}, # OUT OF SCOPE
]

def hit_at(rids, gold, n):  return any(c.split('#')[0]==gold for c in rids[:n])
def mrr(rids, gold):
    for i, c in enumerate(rids, 1):
        if c.split('#')[0]==gold: return 1.0/i
    return 0.0

def evaluate(pipe):
    rows = []
    for e in EVAL:
        out = pipe.answer(e["q"]); rids = [c for c,_ in out["retrieved"]]
        if e["gold"] is None:                              # out-of-scope: should abstain
            ok = "don't know" in out["text"].lower()
            rows.append({**e, "hit": None, "rr": 0.0, "ok": ok, "ans": out["text"][:52]})
        else:
            rows.append({**e, "hit": hit_at(rids, e["gold"], pipe.k), "rr": mrr(rids, e["gold"]),
                         "ok": e["expect"].lower() in out["text"].lower(), "ans": out["text"][:52]})
    return rows

rows = evaluate(rag)
print(f"{'':3}{'hit':6}{'mrr':6} question -> answer")
for r in rows:
    print(f"{'OK ' if r['ok'] else 'XX '}{str(r['hit']):6}{r['rr']:.2f}  {r['q'][:38]:38} -> {r['ans']}")

scoped = [r for r in rows if r["gold"] is not None]
HIT3 = sum(r["hit"] for r in scoped)/len(scoped)
MMRR = sum(r["rr"]  for r in scoped)/len(scoped)
ACC  = sum(r["ok"]  for r in rows)/len(rows)
BASE = sum(1 for e in EVAL if e["expect"].lower() in no_retrieval_answer(e["q"]).lower())/len(EVAL)
print(f"\nRETRIEVAL hit@3={HIT3:.2f}  meanMRR={MMRR:.2f}")
print(f"ANSWER    RAG acc={ACC:.2f}   vs   no-retrieval acc={BASE:.2f}   <-- RAG's whole reason to exist")


## §8 — The four failure modes of naive RAG

RAG (0.70) crushes no-retrieval (0.00) — but it is **not** 1.00, and the misses are not random.
Look at the `XX` rows above. Every naive RAG fails in the same four ways, and **each one is a
Phase 10 lesson.** Let's isolate them with numbers.

### FM1 — The right chunk is retrieved, but not ranked #1  → *reranking (L86)*

A single cosine score is a blunt instrument. A chunk that merely *shares vocabulary* with the
question (e.g. the Support-SLA chunk also says "Pro tier") can out-rank the chunk that actually
answers it. Watch **hit@1** fall below **hit@3**: the gold chunk is *in* the top-k, just not on top
— so the generator reads the wrong one first. That gap is exactly the headroom a **reranker**
recovers, without retrieving a single new document.

In [ ]:
def eval_hits(pipe, n):
    s = [e for e in EVAL if e["gold"]]
    return sum(hit_at([c for c,_ in pipe.answer(e["q"])["retrieved"]], e["gold"], n) for e in s)/len(s)

wide = RagPipeline(DOCS, k=5)
h1, h3, h5 = eval_hits(wide, 1), eval_hits(wide, 3), eval_hits(wide, 5)
print(f"hit@1={h1:.2f}   hit@3={h3:.2f}   hit@5={h5:.2f}")
print(f"reranking headroom (hit@3 - hit@1) = {h3-h1:.2f}  <-- a reranker could recover this for free")
# The gold chunk is already retrieved; it's just mis-ordered. No new documents needed.


### FM2 — The semantic gap: lexical search can't match synonyms  → *dense / hybrid retrieval (L85)*

The question *"how do I reset my password"* returns **nothing useful** — because the doc never says
"password" or "reset". It says "credential recovery" and "set a new secret". TF-IDF matches
**words**, not **meaning**, so it scores this a flat **0.000**. Prove it's purely lexical: feed the
query the doc's *own* vocabulary and similarity jumps. Dense embeddings close this gap because they
encode meaning — synonyms land near each other in vector space.

In [ ]:
raw   = index.search("how do I reset my password", k=1)[0][1]
vocab = index.search("credential recovery set a new secret sign-in link", k=1)[0][1]
print(f"'reset my password'            -> top sim = {raw:.3f}   (lexical miss: no shared words)")
print(f"'credential recovery ... link' -> top sim = {vocab:.3f}   (same doc, matching vocabulary)")
print("\nThe answer was always in the corpus. Lexical retrieval simply could not SEE it.")
# EXPERIMENT: this is the #1 reason teams move from keyword search to embeddings.


### FM3 — A similarity threshold can't reliably tell in-scope from out-of-scope  → *grounding & abstention (L87)*

You might hope "just reject anything below a similarity floor" solves hallucination. It doesn't. An
**out-of-scope** question ("mobile app for iOS") can share enough incidental words with some chunk
to score *higher* than a perfectly **in-scope** question that happened to phrase things differently.
So a single global threshold will either let junk through or reject good queries. Real grounding
needs a **relevance judge** (a reranker score or an LLM checking whether the context actually answers
the question), not one magic number.

In [ ]:
oos     = index.search("does Nimbus offer a mobile app for iOS", k=1)[0][1]  # out of scope
inscope = index.search("how do I reset my password", k=1)[0][1]              # in scope, gap-worded
print(f"OUT-of-scope  'mobile app'      -> sim {oos:.3f}")
print(f"IN-scope      'reset password'  -> sim {inscope:.3f}")
print(f"Out-of-scope OUTSCORES a real question? {oos > inscope}  <-- no single floor separates them")

# The danger with NO grounding floor: it answers an unanswerable question.
r = index.search("does Nimbus offer a mobile app for iOS", k=3)
print("\nmin_sim=0.0 (no floor):", generate("does Nimbus offer a mobile app for iOS", r, min_sim=0.0)["text"][:58], "...")
print("=> confidently returns an irrelevant chunk. This is how RAG systems 'hallucinate'.")


### FM4 — Chunk size is a hyperparameter, not a default  → *chunking strategies (L84)*

The `max_chars=180` we picked was a guess. Sweep it and retrieval accuracy **moves** — too small and
a fact gets split from its context; too large and the real answer is diluted by unrelated sentences
in the same chunk. There is no universal right value; it depends on your documents, and it must be
**measured**.

In [ ]:
print("max_chars   nchunks   hit@1   hit@3")
for mc in (60, 120, 180, 320, 100000):
    p = RagPipeline(DOCS, k=5, max_chars=mc)
    print(f"  {mc:<9} {len(p.chunks):<9} {eval_hits(p,1):.2f}    {eval_hits(p,3):.2f}")
print("\nSame corpus, same retriever, same questions — only the chunk size changed.")
# EXPERIMENT: hit@1 alone swings with max_chars. Lesson 84 makes this principled.


### The failure map

| # | Failure mode | Symptom (measured today) | Fixed in |
|---|---|---|---|
| **FM1** | Right chunk, wrong rank | `hit@1 < hit@3` | **L86** Reranking |
| **FM2** | Semantic gap | `sim = 0.000` on a synonym query | **L85** Dense / hybrid retrieval |
| **FM3** | Threshold can't gate scope | out-of-scope sim > in-scope sim | **L87** Grounding & abstention |
| **FM4** | Chunk size untuned | `hit@1` swings with `max_chars` | **L84** Chunking strategies |

Every one of these is a **retrieval** problem, not a generation problem. That's the lesson of the
opener: when RAG is "bad", **look at what retrieval handed the model** before you touch the prompt.

## §9 — Ten production-RAG pitfalls

1. **Blaming the LLM for a retrieval miss.** If the gold chunk isn't in the top-k, the prompt can't
   save you. Inspect what was retrieved first.
2. **No evaluation set.** Without labeled `question -> gold chunk` pairs you're flying blind and
   "improvements" are vibes. Build the eval set *before* tuning.
3. **Only measuring end-to-end accuracy.** Separate **retrieval** metrics (hit@k, MRR) from
   **answer** metrics — they fail for different reasons and have different fixes.
4. **Lexical-only retrieval in a synonym-rich domain** (FM2). Keyword search silently misses
   paraphrases. Use dense or hybrid retrieval.
5. **Trusting a global similarity threshold to detect scope** (FM3). It can't; use a relevance
   judge / reranker score.
6. **No abstention path.** A RAG system that *never* says "I don't know" will confidently answer
   from junk. Grounded refusal is a feature.
7. **Chunk size chosen once and forgotten** (FM4). It's a measured hyperparameter per corpus.
8. **Chunks with no source metadata.** Store `doc`, offsets, and version so you can cite and later
   re-index incrementally.
9. **Re-embedding the whole corpus on every change.** At scale you index once and update
   incrementally; embedding is the expensive part.
10. **No citations.** If the answer can't point at the chunk it came from, you can't audit it — and
    neither can your users.

## §10 — Verification checklist

Deterministic checks proving every claim in this lesson. All should print PASS.

In [ ]:
checks = []
def check(name, cond):
    checks.append(bool(cond)); print(("PASS " if cond else "FAIL ") + name)

# rebuild clean references
rag  = RagPipeline(DOCS, k=3)
wide = RagPipeline(DOCS, k=5)
rows = evaluate(rag)
scoped = [r for r in rows if r["gold"] is not None]
HIT3 = sum(r["hit"] for r in scoped)/len(scoped)
ACC  = sum(r["ok"]  for r in rows)/len(rows)
BASE = sum(1 for e in EVAL if e["expect"].lower() in no_retrieval_answer(e["q"]).lower())/len(EVAL)
h1, h3 = eval_hits(wide, 1), eval_hits(wide, 3)
raw   = index.search("how do I reset my password", k=1)[0][1]
vocab = index.search("credential recovery set a new secret sign-in link", k=1)[0][1]
oos   = index.search("does Nimbus offer a mobile app for iOS", k=1)[0][1]
sweep = {mc: eval_hits(RagPipeline(DOCS, k=5, max_chars=mc), 1) for mc in (60,120,180,320)}

check("corpus chunked into the expected 24 chunks", len(CHUNKS) == 24)
check("RAG beats no-retrieval by a wide margin", ACC >= 0.6 and ACC - BASE >= 0.5)
check("no-retrieval baseline answers zero private questions", BASE == 0.0)
check("retrieval hit@3 is strong (>=0.85)", HIT3 >= 0.85)
check("FM1 reranking headroom exists (hit@1 < hit@3)", h1 < h3)
check("FM2 semantic gap: raw synonym query ~ 0.0", raw < 0.01)
check("FM2 same doc retrievable with its own vocabulary (>0.4)", vocab > 0.4)
check("FM3 out-of-scope outscores an in-scope query", oos > raw)
check("FM3 abstains when top sim below floor",
      "don't know" in generate("q", index.search("how do I reset my password", 3))["text"].lower())
check("FM4 chunk size changes hit@1 (not one flat value)", len(set(sweep.values())) >= 2)
check("grounded generator abstains on out-of-corpus with a high floor",
      generate("x", index.search("ceo home address", 3), min_sim=0.9)["cites"] == [])

print(f"\n{sum(checks)}/{len(checks)} checks passed")
assert sum(checks) == len(checks), "some checks failed"
print("ALL CHECKS PASSED")


## Summary, homework & what's next

### What you built
A **complete naive RAG pipeline** — chunk, embed, index, retrieve, grounded-generate — and, more
importantly, a **measurement harness** (`hit@k`, `MRR`, answer accuracy) that turns "our RAG feels
off" into specific, reproducible numbers.

### The one idea to keep
> **RAG quality is retrieval quality.** The generator can only be as good as the chunk it's handed.
> When RAG is wrong, look at *what was retrieved* before you touch the prompt.

### The four failure modes you can now name and measure
- **FM1** right chunk, wrong rank -> reranking (L86)
- **FM2** semantic gap -> dense/hybrid retrieval (L85)
- **FM3** threshold can't gate scope -> grounding & abstention (L87)
- **FM4** chunk size untuned -> chunking strategies (L84)

### Homework
1. **Grow the eval set** to 20 questions, including 3 more out-of-scope ones. Does `hit@3` hold?
2. **Swap in real embeddings.** `pip install sentence-transformers`, replace the two
   `TfidfVectorizer` lines with `SentenceTransformer("all-MiniLM-L6-v2").encode(...)`, keep
   `cosine_similarity`. Re-run section 8 FM2 — does the "reset password" query now find the doc?
3. **Add citations to the output.** Make `answer()` return the chunk `text` *and* its `id`, and
   print answers as `"<answer> [source: <id>]"`.
4. **Instrument it.** Emit one structured log per query (question, retrieved ids, top sim, abstained?)
   — reuse the `agent-obs` span idea from Phase 9. You'll want this trail in production.
5. **Break FM3 on purpose.** Find an out-of-scope question that scores *above* 0.3 and write one
   sentence on why a fixed threshold can never be the whole grounding story.

### Next lesson (L84) — **Chunking strategies**
We go deep on FM4: fixed vs recursive vs semantic chunking, overlap, and structure-aware splitting
(headings, tables, code). You'll measure which strategy lifts `hit@1` on this exact eval set — the
first of four principled fixes that turn today's naive baseline into a production retriever.